# Logit Model using Statsmodels

In [7]:
# Load libraries
import pandas as pd
import os
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

In [8]:
# Open toronto_license_data_geocoded.csv in the data folder
listings_df = pd.read_csv(os.path.join('data', 'toronto_listing_data_cleaned_without_hio.csv'))

In [9]:
X = listings_df.drop('legal_listing', axis=1)
y = listings_df['legal_listing']

# Make sure categorical variables are properly represented
X = X.copy()
X = pd.get_dummies(X, drop_first=True)  # if you have categorical vars

problematic_vars = detect_separation(X, y)
print("Potential separation-causing variables:\n", problematic_vars)

# Remove problematic variables from the dataset
listings_df = listings_df.drop(columns=problematic_vars)

Potential separation-causing variables:
 ['neighbourhood_cleansed_Leaside-Bennington', 'neighbourhood_cleansed_Humber Heights-Westmount', 'neighbourhood_cleansed_The Beaches', 'neighbourhood_cleansed_Lambton Baby Point', 'neighbourhood_cleansed_Regent Park', 'neighbourhood_cleansed_North St.James Town', 'neighbourhood_cleansed_Mount Dennis', 'neighbourhood_cleansed_Eringate-Centennial-West Deane', 'neighbourhood_cleansed_Dovercourt-Wallace Emerson-Junction', 'neighbourhood_cleansed_Lawrence Park South', 'neighbourhood_cleansed_Cliffcrest', 'neighbourhood_cleansed_Junction Area', 'neighbourhood_cleansed_Etobicoke West Mall', 'neighbourhood_cleansed_Agincourt South-Malvern West', 'neighbourhood_cleansed_Stonegate-Queensway', 'neighbourhood_cleansed_Bathurst Manor', 'neighbourhood_cleansed_Wexford/Maryvale', 'neighbourhood_cleansed_Pelmo Park-Humberlea', 'neighbourhood_cleansed_University', 'neighbourhood_cleansed_Forest Hill South', 'neighbourhood_cleansed_Yonge-St.Clair', 'neighbourhood

In [10]:
# Load cleaned data
# listings_df = pd.read_csv(os.path.join('data', 'yvr_listing_data_cleaned.csv'))

In [11]:
# Define the independent and dependent variables
X = listings_df.drop('legal_listing', axis=1)
y = listings_df['legal_listing']

In [12]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
# Find all columns in X with a low variance
low_variance = []
for col in X:
    if X[col].var() < 0.001:
        low_variance.append(col)

print(low_variance)

['room_type_Shared room', 'neighbourhood_cleansed_Flemingdon Park', 'neighbourhood_cleansed_Kennedy Park']


In [14]:
# Drop columns with low variance
X = X.drop(low_variance, axis=1)

In [15]:
# Add a constant column to the independent variables
X = sm.add_constant(X_train)

# Create the logit model
logit_model = sm.Logit(y_train, X_train)

In [16]:
# Fit the model
result = logit_model.fit(method='bfgs', maxiter=1000)

c:\Users\Juanes\miniforge3\envs\RentalLicenseModel\lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
c:\Users\Juanes\miniforge3\envs\RentalLicenseModel\lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
c:\Users\Juanes\miniforge3\envs\RentalLicenseModel\lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
c:\Users\Juanes\miniforge3\envs\RentalLicenseModel\lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


Optimization terminated successfully.
         Current function value: 0.270272
         Iterations: 326
         Function evaluations: 339
         Gradient evaluations: 332


In [17]:
# Test its prediction accuracy
predictions = result.predict(X_test)
predictions = (predictions > 0.5).astype(int)
accuracy = (predictions == y_test).mean()
print(f'Accuracy of the model: {accuracy:.4f}')


Accuracy of the model: 0.8947


In [85]:
# Print the summary of the model
print(result.summary())

                           Logit Regression Results                           
Dep. Variable:          legal_listing   No. Observations:                 4177
Model:                          Logit   Df Residuals:                     4105
Method:                           MLE   Df Model:                           71
Date:                Sun, 13 Apr 2025   Pseudo R-squ.:                  0.1243
Time:                        23:57:42   Log-Likelihood:                -1128.9
converged:                       True   LL-Null:                       -1289.1
Covariance Type:            nonrobust   LLR p-value:                 2.229e-33
                                                                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------------------------
estimated_occupancy_l365d                                     -0.0036      0.001     -4.177      0.000      -0.005

In [96]:
# Define independent and dependent variables
X = sm.add_constant(X)  # if not already added
y = listings_df['legal_listing']  # or your target variable
# Ensure X and y have the same index
y = y.loc[X.index]  # or vice versa, as long as they match

# Fit model using GLM with Binomial family
model = sm.GLM(y, X, family=sm.families.Binomial())
results = model.fit()  # HC0, HC1, HC2, HC3

# Show summary with robust SEs
print(results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:          legal_listing   No. Observations:                 5222
Model:                            GLM   Df Residuals:                     5149
Model Family:                Binomial   Df Model:                           72
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1393.0
Date:                Mon, 14 Apr 2025   Deviance:                       2785.9
Time:                        00:10:30   Pearson chi2:                 5.08e+03
No. Iterations:                     7   Pseudo R-squ. (CS):            0.08845
Covariance Type:            nonrobust                                         
                                                                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------

In [95]:
results = model.fit(cov_type='HC3')  # 'HC0', 'HC1', 'HC2', 'HC3' are all options
print(results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:          legal_listing   No. Observations:                 5222
Model:                            GLM   Df Residuals:                     5149
Model Family:                Binomial   Df Model:                           72
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1393.0
Date:                Mon, 14 Apr 2025   Deviance:                       2785.9
Time:                        00:10:16   Pearson chi2:                 5.08e+03
No. Iterations:                     7   Pseudo R-squ. (CS):            0.08845
Covariance Type:                  HC3                                         
                                                                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------

In [4]:
def detect_separation(X, y, threshold=1.0):
    """
    Checks for variables in X that can perfectly or nearly perfectly predict y.
    
    Parameters:
    - X: pd.DataFrame of predictors
    - y: pd.Series of binary target (0/1)
    - threshold: proportion (default=1.0) for 'perfect' separation. Use <1.0 to check for near-separation.
    
    Returns:
    - List of column names likely causing separation
    """
    problematic_cols = []

    for col in X.columns:
        if X[col].nunique() > 50:
            continue  # Skip continuous variables for now

        cross_tab = pd.crosstab(X[col], y, normalize='index')

        for val in cross_tab.index:
            # Look for cases where a category always maps to one outcome
            if (cross_tab.loc[val] == 1).any() or (cross_tab.loc[val] == 0).any():
                max_class_prop = cross_tab.loc[val].max()
                if max_class_prop >= threshold:
                    problematic_cols.append(col)
                    break  # No need to check other values in this column

    return list(set(problematic_cols))

In [41]:
X = listings_df.drop('legal_listing', axis=1)
y = listings_df['legal_listing']

# Make sure categorical variables are properly represented
X = X.copy()
X = pd.get_dummies(X, drop_first=True)  # if you have categorical vars

problematic_vars = detect_separation(X, y)
print("Potential separation-causing variables:\n", problematic_vars)

Potential separation-causing variables:
 ['neighbourhood_cleansed_Kingsview Village-The Westway', 'neighbourhood_cleansed_Scarborough Village', 'neighbourhood_cleansed_Kingsway South', 'neighbourhood_cleansed_Bayview Woods-Steeles', 'neighbourhood_cleansed_East End-Danforth', "neighbourhood_cleansed_L'Amoreaux", 'neighbourhood_cleansed_Woburn', 'neighbourhood_cleansed_Bridle Path-Sunnybrook-York Mills', 'minimum_nights', 'neighbourhood_cleansed_Humber Summit', 'neighbourhood_cleansed_Oakridge', 'neighbourhood_cleansed_Dorset Park', 'neighbourhood_cleansed_Princess-Rosethorn', 'neighbourhood_cleansed_Playter Estates-Danforth', 'number_of_reviews_l30d', 'neighbourhood_cleansed_Humewood-Cedarvale', 'neighbourhood_cleansed_Mount Olive-Silverstone-Jamestown', 'neighbourhood_cleansed_Rockcliffe-Smythe', 'neighbourhood_cleansed_Thorncliffe Park', "neighbourhood_cleansed_O'Connor-Parkview", 'neighbourhood_cleansed_Eglinton East', 'neighbourhood_cleansed_Humbermede', 'neighbourhood_cleansed_Cor

In [43]:
len(problematic_vars)

49

In [75]:
# Print all the coefficients of the model, with their names, only if their p-value is less than 0.05
for i in range(len(result.pvalues)):
    if result.pvalues[i] < 0.05:
        print(result.params.index[i], result.params[i])

host_acceptance_rate 0.009617953227609043
host_identity_verified 1.1299983713279034
host_response_time 0.7873159911939474
instant_bookable -0.4963523595744357
neighbourhood_cleansed_Church-Yonge Corridor -1.6136010085814323
neighbourhood_cleansed_Highland Creek -1.6823896104279257
neighbourhood_cleansed_Hillcrest Village -2.893171331619135
neighbourhood_cleansed_Ionview -1.9280653531258465
neighbourhood_cleansed_Kennedy Park -2.2066125702068087
neighbourhood_cleansed_Kensington-Chinatown -0.6116419340258842
neighbourhood_cleansed_Moss Park -1.2713974947157216
neighbourhood_cleansed_Mount Pleasant East -1.5071599526984194
neighbourhood_cleansed_Mount Pleasant West -2.095676305285298
neighbourhood_cleansed_New Toronto -1.6354528004507103
neighbourhood_cleansed_Newtonbrook East -1.185285890055189
neighbourhood_cleansed_Steeles -1.6465464185395922


C:\Users\Juanes\AppData\Local\Temp\ipykernel_22544\487320113.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if result.pvalues[i] < 0.05:
C:\Users\Juanes\AppData\Local\Temp\ipykernel_22544\487320113.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(result.params.index[i], result.params[i])
